In [9]:
import json
import pandas as pd
import pprint
from pathlib import Path


# Variable for the path data comes from
data_path = Path("../full_data")

# With ensures resources are cleanly released and closed after being used
# Opens the file and reads json into duped_data converting into python objects
with open(data_path/"deduped_training_data.json") as f:
    duped_data = json.load(f)

print(type(duped_data))
print(len(duped_data))

<class 'dict'>
8


In [10]:
# Loops through the 8 keys, checks if it has a length, prints the key and # of 
#   elements within it 
for key, value in duped_data.items():
    size = len(value) if hasattr(value, "__len__") else "n/a"
    print(key, "->", type(value).__name__, size)

policy_id -> dict 5880
segment_id -> dict 5880
annotation_id -> dict 5880
category -> dict 5880
purpose -> dict 5880
does_or_does_not -> dict 5880
entity -> dict 5880
text -> dict 5880


In [20]:
# Showing structure of one row
df = pd.DataFrame(duped_data)
df.iloc[0]

policy_id                                                        3905
segment_id                                                          1
annotation_id                                                   20324
category                                   First Party Collection/Use
purpose                                         Basic service/feature
does_or_does_not                                                 Does
entity                                                            NaN
text                Information that Sci-News.com May Collect Onli...
Name: 0, dtype: object

In [ ]:
# Printing the number of rows in a paragraph to how many paragraphs have that number of rows
counts = df.groupby(["policy_id", "segment_id"]).size()
print(counts.value_counts().sort_index())

1     560
2     597
3     358
4     225
5     121
6      83
7      35
8      32
9      18
10     12
11     10
12      3
13      2
14      2
15      1
17      3
Name: count, dtype: int64


In [ ]:
# Checking for overlap in training/testing files - expected to be 0
train = pd.DataFrame(json.load(open("../full_data/deduped_training_data.json")))
test = pd.DataFrame(json.load(open("../full_data/deduped_testing_data.json")))
print(len(train), len(test))
print(len(set(train["text"]) & set(test["text"])))

5880 1427
33


In [32]:
overlap = set(train["text"]) & set(test["text"])

tr = train[train["text"].isin(overlap)][["text", "policy_id"]]
te = test[test["text"].isin(overlap)][["text", "policy_id"]]

# same text AND same policy in both files = real leakage
merged = tr.merge(te, on=["text", "policy_id"])
print(merged["text"].nunique(), "texts share the same policy across train and test")

# how much of the test set does this affect?
print(test["text"].isin(overlap).mean())

print("Number of policies to train:", train["policy_id"].nunique(), "  Number of Policies to test:", test["policy_id"].nunique())
print("Number of policies that overlap:", len(set(train["policy_id"]) & set(test["policy_id"])))

0 texts share the same policy across train and test
0.060266292922214436
Number of policies to train: 87   Number of Policies to test: 28
Number of policies that overlap: 0
